# Week 1 — 데이터 탐색 + Classical Baseline (2조 Intro)

**구 W1 + W2 통합판** (2026-06-05). 한 세션에 데이터 처리 → sparse 문제 → 단순/Cubic 보간까지.

## 이번 주 학습 목표
1. **데이터 I/O + 전처리** — `.bin` 로드, 정규화, Otsu 임계값
2. **3D voxel 시각화** — 3개 도메인 (BB·CastleGate·Parker) 비교
3. **공극률** + **세 방향 slab 분석**
4. **Sparse imaging** 시뮬레이션 + **단순 평균 보간** + **Cubic spline 보간 (scipy)**
5. **k sweep으로 "왜 deep learning이 필요한가" 정량 확인**

## 학습 방식
**"코드를 직접 짜기보다, 배포된 함수의 인자를 바꿔보며 결과 변화를 관찰하고 해석한다"**

- 각 섹션의 **[Try-it!]** 박스 — 변수 sweep
- **[보조 설명]** — 모르는 용어 풀이
- **[해석 질문]** — 본인 말로 답하기
- 마지막 **탐구 과제**

## 0. 환경 준비

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'helpers'))

import numpy as np
import matplotlib.pyplot as plt

from dr_utils import (
    load_volume, porosity, normalize_to_float, otsu_threshold, binarize_otsu,
    show_slice, show_three_axis, porosity_profile,
    make_sparse, time_saving_ratio, linear_interpolate_slice,
    reconstruct_sparse_linear, reconstruct_sparse_cubic,
    porosity_error, surface_area_error, ssim_3d_mean, summarize_metrics,
    setup_plot_style, ORANGE, NAVY, GREEN, RED, GRAY,
)
setup_plot_style()
print('환경 준비 완료')

## 1. 데이터 로드 — 3 도메인 (BB · CastleGate · Parker)

| 도메인 | 특징 |
|---|---|
| BB | 기준 사암 |
| CastleGate | 고공극·불균질 |
| Parker | 저공극 사암 (W1 신규) |

모두 256×256×256 uint8 binary (0=solid, 1=pore), voxel 2.25 μm.

> **[보조 설명]** **binary segmentation 완료** = 원래의 grayscale CT 영상을 "공극 vs 암석" 두 가지로 분류 완료한 상태. 그래서 값이 0과 1뿐.

In [ ]:
DATA_DIR = Path('..') / 'data'
bb = load_volume(DATA_DIR / 'BB_256.bin')
cg = load_volume(DATA_DIR / 'CastleGate_256.bin')
pk = load_volume(DATA_DIR / 'Parker_256.bin')

for name, v in [('BB', bb), ('CastleGate', cg), ('Parker', pk)]:
    print(f'{name:12s}  shape={v.shape}  dtype={v.dtype}  φ={porosity(v)*100:.2f}%')

## 2. 데이터 전처리 — float 정규화 + Otsu 임계값

Deep learning 모델 학습 전 표준 전처리. W2 이후 매번 필요.

> **[보조 설명: 정규화 (normalize)]** uint8은 0~255 정수, 그런데 deep learning은 [0, 1] 범위 float를 좋아함.
> 그래서 보통 `vol / 255.0` 으로 변환. 학습 안정성이 좋아짐.

> **[보조 설명: Otsu 임계값]** Grayscale 영상에서 "공극과 암석을 가르는 경계값" 을 자동으로 찾는 알고리즘. 두 봉우리 사이 골짜기를 찾음.

In [ ]:
print('--- 정규화 ---')
print(f'원본 BB:  dtype={bb.dtype}, range=[{bb.min()}, {bb.max()}]')
bb_f = normalize_to_float(bb)
print(f'정규화후: dtype={bb_f.dtype}, range=[{bb_f.min()}, {bb_f.max()}]')

print('\n--- Otsu (binary 데이터에서는 trivial — 값이 0/1뿐) ---')
print(f'Otsu threshold of BB z=128: {otsu_threshold(bb_f[128]):.4f}')

> **[Try-it! ①]** Otsu의 진가를 보기 위해 binary 데이터에 노이즈를 일부러 추가해 "가짜 grayscale" 을 만들어봅시다.

In [ ]:
# 가짜 grayscale = binary + 가우시안 노이즈
rng = np.random.default_rng(42)
gray = bb_f[128] + rng.normal(0, 0.2, bb_f[128].shape)
gray = np.clip(gray, 0, 1)

t = otsu_threshold(gray)
binary, _ = binarize_otsu(gray)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(gray); axes[0].set_title('가짜 grayscale (binary + noise)'); axes[0].axis('off')
axes[1].hist(gray.ravel(), bins=80, color=GRAY)
axes[1].axvline(t, color=ORANGE, lw=2, label=f'Otsu t={t:.3f}')
axes[1].set_title('히스토그램 + Otsu'); axes[1].legend()
axes[2].imshow(binary); axes[2].set_title(f'Otsu binarize (φ={binary.mean()*100:.1f}%)'); axes[2].axis('off')
plt.tight_layout(); plt.show()

> **[Try-it! ②]** 위 셀의 `rng.normal(0, 0.2, ...)` 의 `0.2`를 0.05, 0.4 로 바꿔보세요.
> - 노이즈가 작으면 (0.05) Otsu가 정확. 노이즈가 크면 (0.4) Otsu가 어디서 망가지는지 관찰.

## 3. 3 도메인 시각화

본 연구는 "세 방향(z·y·x) 보간 결과 통합 (tri-axis aggregation, W5)" 이 핵심. 왜 세 방향을 모두 보는지 직접 확인.

> **[보조 설명: tri-axis aggregation]** tri = 셋, axis = 축, aggregation = 통합.

In [ ]:
# 3 도메인 중앙 z 슬라이스
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, v) in zip(axes, [('BB', bb), ('CastleGate', cg), ('Parker', pk)]):
    ax.imshow(v[128]); ax.set_title(f'{name} (φ={porosity(v)*100:.1f}%)'); ax.axis('off')
plt.tight_layout(); plt.show()

# BB의 세 축
show_three_axis(bb, title_prefix='BB:')
plt.show()

> **[해석 질문 1]** 세 방향이 "통계적으로는 비슷한 패턴이지만 세부 모양은 다릅니다".
> 만약 z축 한 방향에서만 보간한다면 어떤 문제가 생길 것 같나요?
> 세 방향에서 보간한 결과를 평균낸다면 이 문제가 어떻게 완화될까요?

## 4. Slab별 공극률 + 등방성 검증

In [ ]:
n_slabs = 8
fig, ax = plt.subplots(figsize=(9, 4))
for name, v, col in [('BB', bb, ORANGE), ('CastleGate', cg, NAVY), ('Parker', pk, GREEN)]:
    p = porosity_profile(v, axis=0, n_slabs=n_slabs)
    ax.plot(np.arange(n_slabs), p*100, marker='o', label=f'{name} (std={p.std():.4f})',
            color=col, lw=2)
ax.set_xlabel(f'z방향 슬랩 인덱스 (총 {n_slabs}개)')
ax.set_ylabel('공극률 (%)')
ax.set_title('도메인별 z축 슬랩 공극률 프로파일')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

> **[Try-it! ③]** `n_slabs` 를 4, 16, 32로 바꾸면 곡선이 어떻게 변하나? 왜?
> `axis=0` 을 `axis=1, axis=2` 로 바꿔 세 축 결과가 비슷한지 (= 등방성) 확인하세요.

## 5. Sparse Imaging — 우리 연구의 문제

**문제**: micro-CT 스캔은 시간/비용 비쌈
**해결**: k 슬라이스마다 1개만 측정, 나머지는 컴퓨터로 복원
**핵심 질문**: 얼마나 정확히 복원할 수 있나?

In [ ]:
k = 5
known, missing = make_sparse(bb, k=k, axis=0)
print(f'k={k}: 측정 {len(known)}장, 누락 {len(missing)}장, 시간 절감 {time_saving_ratio(k):.1f}%')

fig, axes = plt.subplots(1, 6, figsize=(13, 3))
for i, z in enumerate(range(60, 66)):
    axes[i].imshow(bb[z])
    axes[i].set_title(f'z={z}\n{"측정" if z in known else "누락"}',
                      color=GREEN if z in known else RED, fontsize=11)
    axes[i].axis('off')
plt.suptitle(f'Sparse k={k}', y=1.05); plt.tight_layout(); plt.show()

## 6. 두 가지 보간 비교 — Linear (B1) vs Cubic (B2)

- **B1 (Linear)**: 두 측정 슬라이스의 "평균을 적당히" 섞음 (직선)
- **B2 (Cubic, scipy)**: 4 슬라이스를 통과하는 부드러운 곡선 (3차 함수)

평가 지표 3종:
- `|Δφ|` 공극률 오차
- `|ΔSA|` 표면적 오차
- `SSIM` 구조 유사도

In [ ]:
vol = bb
k = 5
print(f'BB, k={k} (시간 80% 절감)')
print('-' * 65)
rec_l = reconstruct_sparse_linear(vol, k=k)
summarize_metrics(rec_l, vol, label='B1 Linear')
rec_c = reconstruct_sparse_cubic(vol, k=k)
summarize_metrics(rec_c, vol, label='B2 Cubic')

In [ ]:
# 시각: 원본 vs B1 vs B2 (z=62, 누락 슬라이스)
z_show = 62
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(vol[z_show]); axes[0].set_title(f'원본 z={z_show}')
axes[1].imshow(rec_l[z_show]); axes[1].set_title('B1 Linear')
axes[2].imshow(rec_c[z_show]); axes[2].set_title('B2 Cubic')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

> **[Try-it! ④]** `k` 를 3, 5, 7로 sweep하면서 두 baseline의 세 지표가 어떻게 변하는지 표로 정리. 어떤 패턴이 보이나요?

## 7. 3 도메인 × k sweep — 본 연구의 motivation

이 곡선이 "왜 deep learning이 필요한가" 의 정량 증거. W3에서 UNet 결과를 같은 plot에 겹쳐 볼 것.

In [ ]:
k_list = [2, 3, 5, 7]
results = {}
for name, vol in [('BB', bb), ('CastleGate', cg), ('Parker', pk)]:
    res = {'k': [], 'dphi': [], 'ssim': []}
    for k in k_list:
        rec = reconstruct_sparse_linear(vol, k=k)
        res['k'].append(k)
        res['dphi'].append(porosity_error(rec, vol) * 100)
        res['ssim'].append(ssim_3d_mean(rec, vol))
    results[name] = res
    print(f'  {name} done')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = {'BB': ORANGE, 'CastleGate': NAVY, 'Parker': GREEN}
for name, r in results.items():
    axes[0].plot(r['k'], r['dphi'], marker='o', lw=2, label=name, color=colors[name])
    axes[1].plot(r['k'], r['ssim'], marker='s', lw=2, label=name, color=colors[name])
axes[0].set_xlabel('k'); axes[0].set_ylabel('|Δφ| (%p)'); axes[0].set_title('B1 Linear — 공극률 오차')
axes[1].set_xlabel('k'); axes[1].set_ylabel('SSIM'); axes[1].set_title('B1 Linear — 구조 유사도')
for ax in axes: ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

> **[해석 질문 2]** 어느 도메인이 sparse 보간이 가장 어려운가요? 왜? (힌트: 공극률 vs 구조 복잡도)

## 8. 자기 점검 & W2 예고

### 자기 점검
1. Voxel과 pixel의 차이
2. `normalize_to_float` 가 deep learning에 왜 필요한가
3. Otsu 임계값이 binary 데이터에서 trivial한 이유
4. k=5 시간 절감률과 BB의 B1·B2 |Δφ|
5. 단순 평균(B1) vs Cubic(B2) 차이가 큰 경우는 언제

### W2 예고 — Deep Learning 입문 (mini UNet 학습)
- 본 W1 baseline을 "deep learning이 어디까지 끌어내리나" 직접 확인
- mini UNet (~100K params) 학생 노트북에서 학습 (10/30/60분 옵션)
- `pip install torch torchvision`

---

## 🎯 W1 탐구 과제 (2조)

**과제 1 (필수)**: 3 도메인 × {B1 Linear, B2 Cubic} × k=[2,3,5] 의 9개 조합에 대해 `summarize_metrics` 를 호출해 결과를 표로 정리. 어느 조합이 가장 정확한가요?

**과제 2 (필수)**: [Try-it!] ②의 인공 grayscale에서 노이즈 σ ∈ {0.05, 0.1, 0.2, 0.3, 0.5} 로 sweep. Otsu binarize 결과의 φ를 plot. σ가 클수록 φ 오차가 어떻게 변하나요?

**과제 3 (선택)**: `reconstruct_sparse_linear` 의 `axis` 인자를 0/1/2로 바꿔서 세 축 sparse 보간 결과의 |Δφ|를 비교. 등방성이 좋다면 세 축 결과가 같아야 합니다 — BB는 어떤가요?

**과제 4 (선택, 도전)**: `linear_interpolate_slice` 의 두 슬라이스 간격을 sweep (1, 3, 7, 15, 30). α=0.5 결과를 한 줄로 시각화. 간격이 클수록 "흐릿함" 이 어떻게 보이나요?